# Volume 12, Chapter 4: Units (Python companion)

Exercises first, solutions at the end. We compute fundamental units of real quadratic fields with continued fractions, test the Artin-type bound of Proposition 4.11 on many cubic fields, look at the product $h_KR_K$, search for solutions of Thue's equation $x^3-2y^3=1$, and see why units are too large to write down: the computer-science answer is the *compact representation*. Requires `cypari2` and `numpy`.

In [ ]:
import math, random, time
import cypari2
pari = cypari2.Pari()
pari.allocatemem(4 * 10**8, silent=True)

## Exercises

**P4.1 (★) Fundamental units by continued fractions.** Implement the continued fraction expansion of $\omega=\frac{1+\sqrt d}2$ (or $\sqrt{d/4}$) with exact integer arithmetic on the complete quotients $\frac{P+\sqrt D}{Q}$, and compute the fundamental unit $\varepsilon$ of $\mathbb Q(\sqrt d)$ for every fundamental discriminant $5\le d<2000$, from the convergents at the end of the first period (Volume 1, Chapter 8). Compare with PARI's `quadunit`.

**P4.2 (★★) Artin's bound in practice.** For the cubic fields $\mathbb Q(\theta)$, $\theta^3=\theta+a$ or $\theta^3=m$, with one real embedding, compute the fundamental unit $\eta>1$ (PARI's `bnfinit`) and check $|d_K|<4\eta^3+35$ (Proposition 4.11). How close does $4\eta^3$ come to $|d_K|$?

**P4.3 (★★) Class number times regulator.** For the primes $p\equiv1\pmod4$, $p<20\,000$, compute $h=h(\mathbb Q(\sqrt p))$ and $R=\log\varepsilon$. Plot or tabulate $hR/\sqrt p$. What fraction of these fields has $h=1$? (Chapter 8: $\frac{hR}{\sqrt p}=\frac12L(1,\chi_p)$.)

**P4.4 (★★) Thue's equation.** Check Exercise 4.6 further: among the units $\varepsilon^k$, $|k|\le300$, of $\mathbb Q(\sqrt[3]2)$, find those with no $\sqrt[3]4$ term.

**P4.5 (★★★) CS application: units are huge.** For primes $p\equiv1\pmod 4$ near $10^4$, $10^6$, $10^8$, compute the length of the period of the continued fraction and the number of decimal digits of $\varepsilon$. Explain why writing $\varepsilon$ in the form $\frac{a+b\sqrt p}2$ is hopeless for $p\approx10^{20}$, and how the *compact representation* $\varepsilon=\prod_i\bigl(\frac{\alpha_i}{\beta_i}\bigr)^{2^{i}}$ with small $\alpha_i,\beta_i$ gets around it.

## Solutions

### P4.1
**How to find it.** With $\omega_0=\frac{P_0+\sqrt D}{Q_0}$, the recursion $a_k=\lfloor\omega_k\rfloor$, $P_{k+1}=a_kQ_k-P_k$, $Q_{k+1}=\frac{D-P_{k+1}^2}{Q_k}$ stays in integers when $Q_0\mid D-P_0^2$. The period ends when $(P,Q)$ returns to its first reduced value. The convergent numerators and denominators $p_k,q_k$ then give the unit $p_k-q_k\bar\omega$ (Volume 1, Chapter 8).

In [ ]:
def fundamental_unit(d):
    # d a positive fundamental discriminant; returns (a, b) with eps = (a + b sqrt(d)) / 2.
    if d % 4 == 1:
        D, P, Q = d, 1, 2          # omega = (1 + sqrt d)/2
    else:
        D, P, Q = d // 4, 0, 1     # omega = sqrt(d/4)
    s = math.isqrt(D)
    p0, p1, q0, q1 = 0, 1, 1, 0
    seen = None
    while True:
        a = (P + s) // Q
        p0, p1 = p1, a * p1 + p0
        q0, q1 = q1, a * q1 + q0
        P, Q = a * Q - P, (D - (a * Q - P)**2) // Q
        if seen is None:
            seen = (P, Q)          # first reduced complete quotient
            continue
        if (P, Q) == seen:
            break
    # eps = p0' - q0' * conj(omega) built from the convergent before the repeat
    x, y = p0, q0
    if d % 4 == 1:
        # x - y * (1 - sqrt d)/2 = (2x - y + y sqrt d)/2
        return 2 * x - y, y
    return 2 * x, y              # x + y sqrt(d/4) = (2x + y sqrt d)/2

bad = 0
for d in range(5, 2000):
    if not pari.isfundamental(d):
        continue
    a, b = fundamental_unit(d)
    e = pari.quadunit(d)
    w = pari('quadgen(%d)' % d)
    # compare with PARI: quadunit(d) = u + v*w with w = (d%4==1 ? (1+sqrt d)/2 : sqrt(d/4))
    u, v = int(pari.real(e)), int(pari.imag(e))
    pa, pb = (2 * u + v, v) if d % 4 == 1 else (2 * u, v)
    norm_ok = (a * a - d * b * b) in (4, -4)
    if not norm_ok or (a, b) != (pa, pb):
        bad += 1
print("fields checked; disagreements with quadunit:", bad)

For every fundamental discriminant below $2000$, the continued fraction gives exactly PARI's fundamental unit. (If the code had produced a unit of norm $\pm4$ that is a power of the fundamental one, the comparison would have failed: the first period always gives the fundamental unit.)

### P4.2
**How to find it.** Here $r=1$, so the regulator is $R_K=\log\eta$ for the fundamental unit $\eta>1$: PARI's `bnfinit(f,1).reg` gives $\eta$ directly.

In [ ]:
rows = []
for f in ['x^3-2', 'x^3-3', 'x^3-5', 'x^3-x-1', 'x^3-x-3', 'x^3+x-1', 'x^3-x-5', 'x^3-10', 'x^3-11', 'x^3-x-11']:
    if pari.polsturm(f) != 1:
        continue
    eta = math.exp(float(pari('bnfinit(%s,1).reg' % f)))   # r = 1, so R_K = log(eta)
    d = abs(int(pari('nfinit(%s).disc' % f)))
    rows.append((f, d, eta, 4 * eta**3 + 35))
    assert d < 4 * eta**3 + 35
for f, d, eta, bound in rows:
    print("%-9s |d_K|=%5d  eta=%12.4f  4 eta^3+35=%14.1f  ratio |d_K|/(4eta^3)=%.3f" % (f, d, eta, bound, d / (4 * eta**3)))

The bound holds everywhere, and it is close to sharp for the fields with tiny units: for $x^3-x-1$ ($|d_K|=23$, $\eta\approx1.3247$, the plastic number) $4\eta^3+35\approx44$, and Artin's $4\eta^3+24\approx33$. For pure cubic fields the units are much larger than the bound requires, so the bound proves fundamentality at once, as in Example 4.12.

### P4.3
**How to find it.** `quadclassunit(p)` returns the class number and the regulator together.

In [ ]:
data = []
for p in pari.primes(2300):
    p = int(p)
    if p % 4 != 1 or p > 20000:
        continue
    q = pari.quadclassunit(p)
    h, R = int(q[0]), float(q[3])
    data.append((p, h, R))
h1 = sum(1 for _, h, _ in data if h == 1) / len(data)
print(len(data), "primes; fraction with h = 1:", round(h1, 3))
vals = [h * R / math.sqrt(p) for p, h, R in data]
print("hR/sqrt(p): min %.3f, mean %.3f, max %.3f" % (min(vals), sum(vals) / len(vals), max(vals)))
big = max(data, key=lambda t: t[2])
print("largest regulator: p=%d, h=%d, R=%.1f (sqrt p = %.1f)" % (big[0], big[1], big[2], math.sqrt(big[0])))

About $81\%$ of these fields have class number $1$. Cohen and Lenstra predicted that the proportion tends to $75.446\ldots\%$ for prime discriminants, a conjecture still open (Volume 14); the convergence is slow, and the proportion at this size is still above the limit. Meanwhile $hR/\sqrt p$ stays between about $0.13$ and $2.1$ in this range, with mean near $0.76$: when $h=1$, the regulator must carry the whole size $\sqrt p\,L(1,\chi_p)$, and units have $\sqrt p$ digits in the exponent. Gauss conjectured that infinitely many real quadratic fields have $h=1$; this is still not proved.

### P4.4
**How to find it.** Multiply by $\varepsilon$ or $\varepsilon^{-1}$ in $\mathbb Z[\theta]$ with exact integers; the coefficients grow like $3.85^{|k|}$, which Python handles easily.

In [ ]:
def mul(u, v):
    # (a + b t + c t^2)(a' + b' t + c' t^2) with t^3 = 2
    a, b, c = u; A, B, Cc = v
    return (a*A + 2*(b*Cc + c*B), a*B + b*A + 2*c*Cc, a*Cc + b*B + c*A)

eps, inv = (1, 1, 1), (-1, 1, 0)
hits = []
for step, k0 in ((eps, 1), (inv, -1)):
    u = (1, 0, 0)
    for k in range(1, 301):
        u = mul(u, step)
        if u[2] == 0:
            hits.append((k0 * k, u[0], -u[1]))
print("k with no cube-root-of-4 term (besides k=0):", hits)

Only $k=-1$ (and $k=0$) in this range: the solutions of $x^3-2y^3=1$ are $(1,0)$ and $(-1,-1)$. A finite search proves nothing, of course; Skolem's method shows that the coefficient of $\sqrt[3]4$ in $\varepsilon^k$, viewed as a $p$-adic analytic function of $k$, has at most two zeros (Chapter 6).

### P4.5
**How to find it.** Count the period with the same recursion as in P4.1, and the digits with `len(str(a))`.

In [ ]:
for target in (10**4, 10**6, 10**8):
    p = int(pari.nextprime(target))
    while p % 4 != 1:
        p = int(pari.nextprime(p + 1))
    t = time.perf_counter()
    a, b = fundamental_unit(p)
    dt = time.perf_counter() - t
    print("p=%d: sqrt p=%7.0f, eps has %6d digits, log eps=%10.1f, %.2f s" % (p, math.sqrt(p), len(str(a)), math.log(a) if a < 10**300 else len(str(a)) * math.log(10), dt))

The number of digits of $\varepsilon$ grows roughly like $\sqrt p$ (times a factor $L(1,\chi_p)/h$ that varies from field to field): about $70$ digits for $p\approx10^4$, $600$ for $10^6$, over a thousand for $10^8$ (here $h$ is larger, which makes $\varepsilon$ smaller). For $p\approx10^{20}$ it would be about $10^{10}$ digits, and the continued fraction would need about $10^{10}$ steps.

The way out is not to write $\varepsilon$ down. Its *compact representation* writes it as a product of $O(\log R)$ factors $(\alpha_i/\beta_i)^{2^i}$ with $\alpha_i,\beta_i$ of size polynomial in $\log d$, found by doubling along the continued fraction ("infrastructure", Shanks 1972; Buchmann, Thiel and Williams 1995). With it one can compute $R$ to any precision, test whether a number is a unit, and reduce $\varepsilon$ modulo any $m$, all in time polynomial in $\log d$ and $R$. Subexponential algorithms (Buchmann 1990) compute $R$ itself in time $e^{O(\sqrt{\log d\log\log d})}$, conditionally on GRH, and Hallgren's quantum algorithm in polynomial time. Lenstra's survey (Sources) tells the story.